# Visualização da Area1 em falsa-cor NIR para o CVAT

Converte a imagem original CBERS-4A (4 bandas) em PNG RGB de 8 bits usando **NIR–Vermelho–Verde (4–3–2)**. Preserva largura, altura e posição dos pixels, sem redimensionar, recortar ou reprojetar. O PNG é apenas para visualização e anotação; não substitui o GeoTIFF para análise radiométrica ou georreferenciamento.

**Pré-requisito:** `Area1.tif` em `Meu Drive/CBERS4A/`.


In [ ]:
# ============================================================
# VISUALIZAÇÃO DA AREA1.TIF EM FALSA-COR NIR PARA O CVAT
# Bandas: NIR (4), Vermelho (3), Verde (2)
# Preserva 489 x 657 pixels, sem redimensionar ou reprojetar
# ============================================================

import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

pasta_base = '/content/drive/MyDrive/CBERS4A'
caminho_tif = os.path.join(pasta_base, 'Area1.tif')
pasta_saida = os.path.join(pasta_base, 'visualizacoes_Area1')
os.makedirs(pasta_saida, exist_ok=True)

with rasterio.open(caminho_tif) as src:
    if src.count < 4:
        raise ValueError('A Area1.tif deve possuir pelo menos quatro bandas.')
    # 2=verde, 3=vermelho, 4=NIR
    verde_original, vermelho_original, nir_original = src.read([2, 3, 4]).astype(np.float32)
    mascara = np.all(src.read_masks([2, 3, 4]) > 0, axis=0)
    altura, largura = src.height, src.width
    print('Imagem original:', largura, 'x', altura, 'pixels')
    print('CRS de referência:', src.crs)

def realcar(banda, mascara, p_baixo=1, p_alto=99, gamma=0.85):
    validos = mascara & np.isfinite(banda)
    valores = banda[validos]
    resultado = np.zeros(banda.shape, dtype=np.float32)
    if valores.size == 0:
        return resultado
    minimo, maximo = np.percentile(valores, [p_baixo, p_alto])
    if maximo <= minimo:
        return resultado
    normalizada = np.clip((banda[validos] - minimo) / (maximo - minimo), 0, 1)
    resultado[validos] = normalizada ** gamma
    return resultado

nir = realcar(nir_original, mascara)
vermelho = realcar(vermelho_original, mascara)
verde = realcar(verde_original, mascara)

# Falsa-cor: NIR no canal R, vermelho no G, verde no B
falsa_cor = np.dstack([nir, vermelho, verde])
pixels_8bit = np.round(np.clip(falsa_cor, 0, 1) * 255).astype(np.uint8)
imagem_nir = Image.fromarray(pixels_8bit)

nome_arquivo = 'Area1_FalsaCor_NIR.png'
caminho_png = os.path.join(pasta_saida, nome_arquivo)
imagem_nir.save(caminho_png, format='PNG', compress_level=6)

assert imagem_nir.size == (largura, altura), 'Dimensões alteradas inesperadamente.'

print('PNG salvo em:', caminho_png)
print('Dimensões do PNG:', imagem_nir.size)
print('Pixels originais mantidos na mesma posição: sim')

# Exibir apenas UMA imagem, sem distorção da proporção
fig, ax = plt.subplots(figsize=(7, 9.5))
ax.imshow(imagem_nir, aspect='equal', interpolation='nearest')
ax.set_title('Area1 — Falsa-cor NIR (bandas 4-3-2)')
ax.axis('off')
plt.tight_layout()
plt.show()
